# Debezium Lakehouse Integration

This notebook demonstrates how to query data captured by Debezium and stored in Apache Iceberg tables.

In [1]:
from pyspark.sql import SparkSession
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))
spark = (
    SparkSession.builder
        .appName("IcebergRestExample")
        .config("spark.sql.catalog.default", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.default.type", "rest")
        .config("spark.sql.catalog.default.uri", "http://rest:8181")
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.iceberg.vectorization.enabled", "false")
        .config("spark.sql.catalog.default.s3.endpoint", "http://minio:9000")
        .config("spark.sql.catalog.default.s3.access-key-id", "admin")
        .config("spark.sql.catalog.default.s3.secret-access-key", "password")
        .config("spark.sql.catalog.default.s3.path-style-access", "true")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.access.key", "admin")
        .config("spark.hadoop.fs.s3a.secret.key", "password")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")   # use http
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider","org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .getOrCreate()
)

spark.sparkContext.applicationId

25/10/13 03:48:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


'local-1760327338467'

In [6]:
# spark = (SparkSession.builder
#     .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
#     .config("spark.sql.catalog.default", "org.apache.iceberg.spark.SparkCatalog")
#     .config("spark.sql.catalog.default.type", "hadoop")
#     .config("spark.sql.catalog.default.warehouse", "s3a://warehouse/")
#     .config("spark.sql.catalog.default.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
#     .getOrCreate()
# )

spark.sql("""
 CALL default.system.remove_orphan_files(table => 'db.orders', dry_run => true)
--CALL default.system.expire_snapshots('db.orders', TIMESTAMP '2025-10-13 00:00:00.000', 10)
""")
# spark.sql("select * from db.orders where id is not null").show(10, False)


Py4JJavaError: An error occurred while calling o52.sql.
: java.io.UncheckedIOException: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.iceberg.spark.actions.DeleteOrphanFilesSparkAction.listDirRecursively(DeleteOrphanFilesSparkAction.java:386)
	at org.apache.iceberg.spark.actions.DeleteOrphanFilesSparkAction.listedFileDS(DeleteOrphanFilesSparkAction.java:311)
	at org.apache.iceberg.spark.actions.DeleteOrphanFilesSparkAction.actualFileIdentDS(DeleteOrphanFilesSparkAction.java:296)
	at org.apache.iceberg.spark.actions.DeleteOrphanFilesSparkAction.doExecute(DeleteOrphanFilesSparkAction.java:247)
	at org.apache.iceberg.spark.JobGroupUtils.withJobGroupInfo(JobGroupUtils.java:59)
	at org.apache.iceberg.spark.JobGroupUtils.withJobGroupInfo(JobGroupUtils.java:51)
	at org.apache.iceberg.spark.actions.BaseSparkAction.withJobGroupInfo(BaseSparkAction.java:130)
	at org.apache.iceberg.spark.actions.DeleteOrphanFilesSparkAction.execute(DeleteOrphanF

In [5]:
spark.sql("""
SELECT * FROM db.orders where id is not null

""").show(10,False)

+----+-----------+----------------+-------+------------+--------------------------+----------+
|id  |customer_id|order_date      |status |total_amount|order_timestamp           |is_deleted|
+----+-----------+----------------+-------+------------+--------------------------+----------+
|1719|1719       |1760279790397069|pending|6709.49     |2025-10-12 14:36:30.397069|false     |
|1716|1716       |1760279777122778|pending|7340.17     |2025-10-12 14:36:17.122778|false     |
|1712|1712       |1760279762759356|pending|11829.83    |2025-10-12 14:36:02.759356|false     |
|1711|1711       |1760279758021432|pending|4610.40     |2025-10-12 14:35:58.021432|false     |
|1713|1713       |1760279765690162|pending|625.56      |2025-10-12 14:36:05.690162|false     |
|1718|1718       |1760279785857502|pending|8872.70     |2025-10-12 14:36:25.857502|false     |
|1720|1720       |1760279793866509|pending|2473.73     |2025-10-12 14:36:33.866509|false     |
|1715|1715       |1760279773418371|pending|2395.86

In [7]:
spark.sql("""
SELECT * FROM db.orders.data_files
""").show(10, False)

+-------+---------------------------------------------------------------------------------------------+-----------+-------+------------+------------------+---------------------------------------------------------------+--------------------------------------------------------+--------------------------------------------------------+----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------+------------+-------------+------------+--------------------+--------------+---------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Query Customers Table

The Spark session is already configured to use the REST Catalog, so we can directly query the tables created by the Iceberg Connect Sink.

We can use the Jupyter magic to make the query too:

In [4]:
%%sql

SELECT * FROM my_database.customers_table LIMIT 10

25/10/12 07:18:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

id,customer_id,order_date,status,total_amount,__deleted,__op,__table,__lsn,__source_ts_ms,updated_at,last_name,created_at,first_name,email,quantity,product_id,unit_price,order_id,price,name,description,category
1,1,1760239570680780,pending,3260.50,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
2,2,1760239575159190,pending,467.47,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
3,3,1760239578379919,pending,902.86,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
4,4,1760239580897594,pending,8263.76,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
5,5,1760239583063753,pending,4251.64,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
6,6,1760239586895835,pending,4889.20,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
7,7,1760239590716165,pending,2846.49,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
8,8,1760239594731975,pending,1706.27,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
9,9,1760239599500800,pending,7284.00,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
10,10,1760239604045214,pending,1402.41,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None


## Query Products Table

Now let's query the products table:

In [5]:
%%sql

SELECT * FROM my_database.products_table LIMIT 10

id,customer_id,order_date,status,total_amount,__deleted,__op,__table,__lsn,__source_ts_ms,updated_at,last_name,created_at,first_name,email,quantity,product_id,unit_price,order_id,price,name,description,category
1,1,1760239570680780,pending,3260.50,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
2,2,1760239575159190,pending,467.47,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
3,3,1760239578379919,pending,902.86,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
4,4,1760239580897594,pending,8263.76,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
5,5,1760239583063753,pending,4251.64,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
6,6,1760239586895835,pending,4889.20,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
7,7,1760239590716165,pending,2846.49,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
8,8,1760239594731975,pending,1706.27,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
9,9,1760239599500800,pending,7284.00,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
10,10,1760239604045214,pending,1402.41,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None


## Query Orders Table

Let's examine the orders table:

In [6]:
%%sql

SELECT * FROM my_database.orders_table LIMIT 10

id,customer_id,order_date,status,total_amount,__deleted,__op,__table,__lsn,__source_ts_ms,updated_at,last_name,created_at,first_name,email,quantity,product_id,unit_price,order_id,price,name,description,category
1,1,1760239570680780,pending,3260.50,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
2,2,1760239575159190,pending,467.47,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
3,3,1760239578379919,pending,902.86,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
4,4,1760239580897594,pending,8263.76,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
5,5,1760239583063753,pending,4251.64,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
6,6,1760239586895835,pending,4889.20,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
7,7,1760239590716165,pending,2846.49,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
8,8,1760239594731975,pending,1706.27,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
9,9,1760239599500800,pending,7284.00,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
10,10,1760239604045214,pending,1402.41,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None


## Query Order Items Table

Finally, let's look at the order items table:

In [7]:
%%sql

SELECT * FROM my_database.order_items_table LIMIT 10

id,customer_id,order_date,status,total_amount,__deleted,__op,__table,__lsn,__source_ts_ms,updated_at,last_name,created_at,first_name,email,quantity,product_id,unit_price,order_id,price,name,description,category
1,1,1760239570680780,pending,3260.50,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
2,2,1760239575159190,pending,467.47,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
3,3,1760239578379919,pending,902.86,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
4,4,1760239580897594,pending,8263.76,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
5,5,1760239583063753,pending,4251.64,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
6,6,1760239586895835,pending,4889.20,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
7,7,1760239590716165,pending,2846.49,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
8,8,1760239594731975,pending,1706.27,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
9,9,1760239599500800,pending,7284.00,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None
10,10,1760239604045214,pending,1402.41,false,r,orders,35825048,1760253361731,None,None,None,None,None,None,None,None,None,None,None,None,None


## Join Tables for Analysis

Now let's perform a more complex query joining multiple tables:

In [ ]:
%%sql

SELECT 
    c.id as customer_id, 
    c.first_name, 
    c.last_name, 
    o.id as order_id,
    o.order_date,
    p.name as product_name,
    oi.quantity,
    oi.unit_price,
    (oi.quantity * oi.unit_price) as total_price
FROM 
    my_database.customers_table c
JOIN 
    my_database.orders_table o ON o.customer_id = c.id
JOIN 
    my_database.order_items_table oi ON oi.order_id = o.id
JOIN 
    my_database.products_table p ON p.id = oi.product_id
LIMIT 10

## Analyze Customer Spending

Let's calculate total spending by customer:

In [ ]:
%%sql

SELECT 
    c.first_name || ' ' || c.last_name as customer_name,
    COUNT(DISTINCT o.id) as order_count,
    SUM(oi.quantity * oi.unit_price) as total_spent
FROM 
    my_database.customers_table c
JOIN 
    my_database.orders_table o ON o.customer_id = c.id
JOIN 
    my_database.order_items_table oi ON oi.order_id = o.id
GROUP BY 
    c.first_name, c.last_name
ORDER BY 
    total_spent DESC
LIMIT 10